# Día 7 · Unidad 7 — pandas y NumPy (I): fundamentos de NumPy

**Objetivos de aprendizaje**
- Explicar qué es un array de NumPy y en qué se diferencia de una lista de Python.
- Crear arrays desde listas, rangos y generadores aleatorios, y leer sus atributos básicos (`shape`, `dtype`, `ndim`).
- Aplicar operaciones vectorizadas y entender *broadcasting* básico.
- Entender por qué pandas (que veremos en el resto del día) se apoya en NumPy por dentro.

**Duración estimada:** 45-60 minutos.

**Requisitos previos:** Día 2 (listas), Día 3 (control de flujo).

**Nota:** este notebook **no** es un curso completo de NumPy — es lo justo para entender por qué pandas funciona como funciona y para no quedarte en blanco si ves `np.algo` en el resto del curso. El programa de la Unidad 7 nombra "pandas y NumPy" juntos precisamente por esto: casi todo lo que hace rápido a pandas por dentro es NumPy.

## 1. Qué es NumPy y por qué existe

[NumPy](https://numpy.org/) (*Numerical Python*) es la librería base de computación numérica en Python. Su pieza central es el `ndarray` (*n-dimensional array*): una estructura de datos parecida a una lista, pero con dos diferencias que lo cambian todo:

1. **Todos los elementos son del mismo tipo** (todo `int`, todo `float`...), a diferencia de una lista de Python, que puede mezclar tipos libremente.
2. **Las operaciones se aplican a todo el array de golpe** (*vectorización*), en vez de tener que escribir un bucle `for` elemento a elemento.

Esa homogeneidad de tipo es lo que permite a NumPy guardar los datos en un bloque de memoria contiguo y delegar las operaciones en código C compilado por debajo — mucho más rápido que un bucle interpretado de Python. pandas está construido sobre NumPy: cada columna de un DataFrame es, por dentro, un array de NumPy (o algo muy parecido).

In [ ]:
import numpy as np

primas = np.array([420.0, 310.5, 275.0, 590.25, 180.0])
print(primas)
print(type(primas))

## 2. Crear arrays

Las formas más habituales de crear un array:

In [ ]:
# Desde una lista de Python
edades = np.array([34, 51, 28, 45, 60])
print("Desde lista:", edades)

# Rango de valores, como range() pero devuelve un array
anios = np.arange(2021, 2025)
print("np.arange:", anios)

# Array de ceros / unos con una forma (shape) dada
matriz_ceros = np.zeros((2, 3))
print("np.zeros:\n", matriz_ceros)

# Valores equiespaciados entre dos límites
franjas_edad = np.linspace(18, 65, num=5)
print("np.linspace:", franjas_edad)

# Números aleatorios (con semilla fija para que el notebook sea reproducible)
rng = np.random.default_rng(seed=42)
siniestros_simulados = rng.normal(loc=1500, scale=400, size=6)
print("np.random:", np.round(siniestros_simulados, 2))

## 3. Atributos básicos

Un array tiene atributos que describen su forma y contenido — no son métodos, se leen sin paréntesis, igual que verás luego con pandas.

In [ ]:
cartera = np.array([[1, 25000.0], [2, 18500.0], [3, 42000.0]])

print("shape (filas, columnas):", cartera.shape)
print("ndim (número de dimensiones):", cartera.ndim)
print("dtype (tipo de dato):", cartera.dtype)
print("size (número total de elementos):", cartera.size)

> ⚠️ **Error típico**: mezclar tipos al crear un array esperando que se comporten como en una lista. `np.array([1, "auto", 3.5])` no lanza error, pero NumPy convierte **todo** al tipo más general posible (aquí, texto): pierdes la posibilidad de hacer aritmética con esos valores sin darte cuenta, porque ya no son números.

In [ ]:
mezcla = np.array([1, "auto", 3.5])
print(mezcla)
print(mezcla.dtype)  # '<U32' o similar: todo se convirtió a texto

try:
    mezcla + 1
except TypeError as e:
    print("Error esperado:", e)

## 4. Operaciones vectorizadas: por qué importa

Con una lista de Python, para aplicar el 8% de Impuesto sobre Primas de Seguro (IPS) a una lista de primas necesitas iterar elemento a elemento:

In [ ]:
primas_lista = [420.0, 310.5, 275.0, 590.25, 180.0]

primas_con_ips_lista = []
for prima in primas_lista:
    primas_con_ips_lista.append(round(prima * 1.08, 2))

print(primas_con_ips_lista)

Con un array de NumPy, la misma operación se escribe **sin bucle**: se aplica a todos los elementos a la vez.

In [ ]:
primas_array = np.array(primas_lista)
primas_con_ips_array = np.round(primas_array * 1.08, 2)

print(primas_con_ips_array)

Ambas dan el mismo resultado, pero la versión vectorizada es más corta, más difícil de romper (no hay índice que se te pueda desalinear) y, con volúmenes grandes de datos, muchísimo más rápida — porque NumPy delega el bucle en código compilado en vez de interpretarlo línea a línea en Python.

> 📊 **En Excel esto sería...** como arrastrar una fórmula (`=A2*1.08`) por toda una columna de una vez, en vez de escribir el cálculo celda a celda a mano: el motor de cálculo de Excel aplica la misma operación a todo el rango de golpe.

**Para ti:** crea un array `franquicias = np.array([300, 150, 0, 600, 450])` y calcula, sin usar ningún bucle `for`, un array `franquicias_con_descuento` que sea cada franquicia con un 10% de descuento, redondeado a 2 decimales.

In [ ]:
# Tu código aquí
franquicias = np.array([300, 150, 0, 600, 450])


### Comparación de tiempos (para que lo veas de verdad, no solo que te lo digan)

Con pocos elementos la diferencia es imperceptible, pero con un volumen realista de datos (por ejemplo, una cartera de 500.000 pólizas) se nota mucho.

In [ ]:
import time

n = 500_000
primas_grandes_lista = [round(100 + i * 0.01, 2) for i in range(n)]
primas_grandes_array = np.array(primas_grandes_lista)

inicio = time.perf_counter()
resultado_bucle = [round(p * 1.08, 2) for p in primas_grandes_lista]
tiempo_bucle = time.perf_counter() - inicio

inicio = time.perf_counter()
resultado_vectorizado = np.round(primas_grandes_array * 1.08, 2)
tiempo_vectorizado = time.perf_counter() - inicio

print(f"Bucle de Python:   {tiempo_bucle:.4f} s")
print(f"NumPy vectorizado: {tiempo_vectorizado:.4f} s")
print(f"NumPy es aproximadamente {tiempo_bucle / tiempo_vectorizado:.0f}x más rápido en esta máquina")

El factor exacto depende de la máquina donde ejecutes esto, pero la idea de fondo se mantiene siempre: **evita bucles `for` sobre arrays o columnas de pandas cuando existe una operación vectorizada equivalente.** Es la razón de fondo por la que, en el resto del curso, verás muy pocos bucles `for` sobre datos de pandas.

## 5. Broadcasting básico

*Broadcasting* es la regla que sigue NumPy para operar entre arrays de formas distintas sin que tengas que igualar manualmente sus tamaños. El caso más simple, que ya usaste arriba sin nombrarlo, es operar un array con un único número (un *escalar*): NumPy "expande" el escalar para que encaje con cada elemento.

In [ ]:
primas_base = np.array([420.0, 310.5, 275.0])
print(primas_base + 50)       # suma 50 a cada elemento
print(primas_base * 1.08)     # multiplica cada elemento por 1.08

También funciona entre un array 2D y un array 1D, siempre que sus formas sean compatibles — aquí, una fila de recargos por producto se aplica a cada fila de una matriz de primas por (póliza, producto):

In [ ]:
# 3 pólizas x 2 productos (AUTO, HOGAR)
primas_matriz = np.array([
    [400.0, 200.0],
    [350.0, 180.0],
    [500.0, 250.0],
])

recargo_por_producto = np.array([1.05, 1.02])  # 5% AUTO, 2% HOGAR

primas_con_recargo = primas_matriz * recargo_por_producto
print(primas_con_recargo)

NumPy "repite" `recargo_por_producto` para cada una de las 3 filas sin que tengas que construir una matriz de 3x2 a mano. La regla completa de broadcasting tiene más casos (arrays de más dimensiones, formas parcialmente compatibles...), pero no hace falta profundizar más para este curso: lo importante es reconocer el patrón cuando lo veas en pandas, porque una Serie de pandas operada con un único número sigue exactamente esta misma lógica.

> ⚠️ **Error típico**: intentar operar dos arrays de formas incompatibles esperando que NumPy "adivine" lo que quieres. Si las formas no encajan según la regla de broadcasting, NumPy lanza un error en vez de dar un resultado silenciosamente incorrecto — lo cual, aquí, es una ventaja: falla alto y claro en vez de fallar en silencio.

In [ ]:
try:
    np.array([1, 2, 3]) + np.array([1, 2])
except ValueError as e:
    print("Error esperado:", e)

## 6. Indexado, slicing y filtros: comparación directa con listas

Ya conoces el *slicing* de listas (`lista[1:3]`). En NumPy funciona igual, y además añade el filtrado por condición booleana que verás constantemente en pandas.

In [ ]:
edades = np.array([34, 51, 28, 45, 60, 22, 39])

print("Slicing (igual que en listas):", edades[1:4])
print("Últimos dos elementos:", edades[-2:])

# Filtro booleano: NO existe en listas de Python de forma nativa
mascara = edades >= 45
print("Máscara booleana:", mascara)
print("Edades >= 45:", edades[mascara])

# En una sola línea, es el patrón que usarás sin parar en pandas
print("Directo:", edades[edades >= 45])

Con una lista de Python, el equivalente sería un bucle o una *list comprehension*:

```python
edades_lista = [34, 51, 28, 45, 60, 22, 39]
mayores_45 = [e for e in edades_lista if e >= 45]
```

Funciona igual de bien para este caso simple, pero no se generaliza tan directamente a tablas de datos con muchas columnas — que es exactamente el problema que resuelve pandas apoyándose en este mecanismo de NumPy.

**Para ti:** con el array `siniestros_simulados` que creaste en la sección 2, filtra y guarda en `siniestros_altos` solo los valores superiores a 1800. ¿Cuántos hay? (pista: `len(...)` o `.size`).

In [ ]:
# Tu código aquí


## Resumen

- Un array de NumPy (`ndarray`) es homogéneo en tipo (a diferencia de una lista de Python) y eso es lo que permite operarlo en bloque, sin bucles `for` explícitos — se llama **vectorización**.
- Se crea con `np.array()`, `np.arange()`, `np.zeros()`/`np.ones()`, `np.linspace()` o generadores aleatorios (`np.random.default_rng()`).
- Sus atributos básicos son `shape`, `ndim`, `dtype` y `size` — se leen sin paréntesis.
- Las operaciones vectorizadas (`array * 1.08`, `np.round(...)`) son más cortas y, con volúmenes grandes de datos, mucho más rápidas que el bucle equivalente.
- **Broadcasting** es la regla que permite operar arrays de formas distintas (por ejemplo, un array y un único número) sin igualarlos manualmente; si las formas son incompatibles, NumPy lanza un error claro en vez de dar un resultado silenciosamente incorrecto.
- El filtrado por condición booleana (`array[array > umbral]`) es el mismo mecanismo que usarás constantemente al filtrar DataFrames de pandas — merece la pena que se quede fijado ya en NumPy, antes de complicarlo con columnas y tipos mixtos.
- Cada columna de un DataFrame de pandas es, por dentro, esencialmente un array de NumPy — por eso el programa de esta unidad los nombra juntos, y por eso este notebook va primero.

**Siguiente:** `02_teoria_series.ipynb`, donde veremos la primera estructura propia de pandas: la Serie.